# cifake-robustness on Colab

Use a GPU runtime (Runtime → Change runtime type → GPU). Re-scoring two detectors takes a few minutes; retraining one takes about half an hour.


## Code and data

In [ ]:
!git clone https://github.com/thisisdawit/cifake-robustness.git
%cd cifake-robustness
!pip -q install -r requirements.txt kagglehub huggingface_hub

## The models

All three detectors are defined in `src/models.py`. The Small CNN is written from scratch for 32×32 inputs; ResNet-50 and EfficientNet-B0 are ImageNet-pretrained backbones with a new one-output head, either frozen or fine-tuned.

In [ ]:
import inspect
from src import models
print(inspect.getsource(models.build_small_cnn))
models.build_small_cnn().summary()

In [ ]:
print(inspect.getsource(models.build_transfer))

Hardened detectors see each training image randomly blurred, resized, noised and JPEG-compressed every time it is drawn:

In [ ]:
from src import transforms
print(inspect.getsource(transforms.tf_training_degradation))

In [ ]:
import kagglehub
# Public dataset; if a login is requested, run kagglehub.login() with a Kaggle API token first.
DATA = kagglehub.dataset_download('birdy654/cifake-real-and-ai-generated-synthetic-images')
print(DATA)

## Trained weights

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('thisisdawit/cifake-robustness', local_dir='models', allow_patterns=['*.weights.h5'])
!ls models

## Re-score the baseline and hardened Small CNN under all sixteen conditions

Add any other tag from `models/` (with its `--model`) to re-score it as well.

In [ ]:
for tag in ['small_cnn', 'small_cnn_hardened']:
    !python -m src.evaluate --root "$DATA" --weights models/{tag}.weights.h5 --model small_cnn --tag {tag} --outdir rescored

## Compare with the stored results

In [ ]:
import json
for tag in ['small_cnn', 'small_cnn_hardened']:
    new = json.load(open(f'rescored/audit_{tag}.json'))['conditions']
    old = json.load(open(f'results/audit_{tag}.json'))['conditions']
    gap = max(abs(new[c][m] - old[c][m]) for c in new for m in ('accuracy', 'auc', 'ece'))
    print(f'{tag:20} accuracy clean {new["clean"]["accuracy"]:.4f}, resize+JPEG {new["social_q50"]["accuracy"]:.4f}; largest difference from results/: {gap:.5f}')

## Retrain the hardened Small CNN

A new training run agrees with the stored one within run-to-run variation, not exactly.

In [ ]:
!python -m src.train --root "$DATA" --dedup --model small_cnn --epochs 30 --harden --tag small_cnn_hardened_retrained --outdir rescored --modeldir rescored
!python -m src.evaluate --root "$DATA" --weights rescored/small_cnn_hardened_retrained.weights.h5 --model small_cnn --tag small_cnn_hardened_retrained --outdir rescored